# DNN PyTorch — Allocation Intelligente de Patients

Modèle pair-wise : score(patient, hôpital) → hôpital optimal avec masque de faisabilité.

## 1. Imports & Configuration

Chargement de toutes les bibliothèques nécessaires :
- **PyTorch / torch.nn** : construction et entraînement du réseau de neurones
- **scikit-learn** : normalisation (`StandardScaler`), encodage des labels, split train/test
- **pandas / numpy** : manipulation des données tabulaires
- **matplotlib / seaborn** : visualisation des courbes d'entraînement

> `DEVICE` détecte automatiquement si un GPU (CUDA) est disponible, sinon utilise le CPU.

In [3]:
import math, pickle, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

ROOT         = Path.cwd()
PATIENTS_XLS = ROOT.parent / 'xgboost_model' / 'patients_1000_ULTRA_COMPLET.xlsx'
HOSPIT_XLS   = ROOT / 'Book1.xlsx'
FEASIB_CSV   = ROOT / 'output' / 'feasibility_matrix.csv'
OUT_DIR      = ROOT / 'output'
OUT_DIR.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

ModuleNotFoundError: No module named 'torch'

## 2. Constantes & Mapping des Ressources

Définition des colonnes utilisées comme **features d'entrée du DNN** :

| Groupe | Colonnes | Nb |
| --- | --- | --- |
| Patient | ESI, vitaux (SpO2, FC, FR, GCS), besoins médicaux, GPS | 31 |
| Hôpital | Capacités en personnel, équipements, lits | 22 |
| Distance | Distance Haversine patient→hôpital (km) | 1 |
| **Total** | | **54** |

`RESOURCE_MAPPINGS` fait le lien entre le nom d'une colonne côté **patient** et son équivalent côté **hôpital**.

In [ ]:
PATIENT_FEATURES = [
    'ESI','SpO2','FC','FR','GCS',
    'O2','Ventilateur','Sang','Defibrillateur','Monitoring',
    'Reanimation','Labo','Imagerie',
    'Medecins','Infirmiers','Urgentistes','Reanimateurs','Anesth_Rea',
    'Pneumo','Cardio','Neuro','Internistes','Chirurgiens','Pediatres',
    'Biologistes','Radiologues','Moniteurs','Rea_lits','Lits_totaux',
    'Latitude','Longitude'
]  # 31 features

HOSP_RES_COLS = [
    'Médecins','Infirmiers','Urgentistes','Réanimateurs','Anesth.-Réa',
    'Pneumo','Cardio','Neuro','Internistes','Chirurgiens','Pédiatres',
    'Biologistes','Radiologues','Labo','Imagerie',
    'O\u2082','Ventilateurs','Moniteurs','Défibrillateurs','Sang (unités)',
    'Réa (lits)','Lits totaux'
]  # 22 features  => + distance = 23

RESOURCE_MAPPINGS = {
    'Lit':{'p':'Lits_totaux','h':'Lits totaux'},
    'Rea_Lit':{'p':'Rea_lits','h':'Réa (lits)'},
    'Medecin':{'p':'Medecins','h':'Médecins'},
    'Infirmier':{'p':'Infirmiers','h':'Infirmiers'},
    'Urgentiste':{'p':'Urgentistes','h':'Urgentistes'},
    'Reanimateur':{'p':'Reanimateurs','h':'Réanimateurs'},
    'Anesth_Rea':{'p':'Anesth_Rea','h':'Anesth.-Réa'},
    'Pneumo':{'p':'Pneumo','h':'Pneumo'},
    'Cardio':{'p':'Cardio','h':'Cardio'},
    'Neuro':{'p':'Neuro','h':'Neuro'},
    'Interniste':{'p':'Internistes','h':'Internistes'},
    'Chirurgien':{'p':'Chirurgiens','h':'Chirurgiens'},
    'Pediatre':{'p':'Pediatres','h':'Pédiatres'},
    'Biologiste':{'p':'Biologistes','h':'Biologistes'},
    'Radiologue':{'p':'Radiologues','h':'Radiologues'},
    'O2':{'p':'O2','h':'O\u2082'},
    'Ventilateur':{'p':'Ventilateur','h':'Ventilateurs'},
    'Sang':{'p':'Sang','h':'Sang (unités)'},
    'Moniteur':{'p':'Moniteurs','h':'Moniteurs'},
    'Defibrillateur':{'p':'Defibrillateur','h':'Défibrillateurs'},
    'Monitoring':{'p':'Monitoring','h':'Moniteurs'},
    'Labo':{'p':'Labo','h':'Labo'},
    'Imagerie':{'p':'Imagerie','h':'Imagerie'},
}

INPUT_DIM = len(PATIENT_FEATURES) + len(HOSP_RES_COLS) + 1  # 31+22+1=54
print(f'Input dim DNN: {INPUT_DIM}')

## 3. Chargement des Données

Trois sources de données sont chargées :
- `patients_1000_ULTRA_COMPLET.xlsx` — 1 000 patients avec leurs besoins médicaux et coordonnées GPS
- `Book1.xlsx` — 22 hôpitaux avec leurs capacités en ressources
- `feasibility_matrix.csv` — matrice booléenne (1000×22) pré-calculée indiquant quels hôpitaux
  peuvent **physiquement** accueillir chaque patient (calculée dans `build_feasibility.ipynb`)

> Le nettoyage numérique (`pd.to_numeric`) garantit qu'aucune valeur texte ne perturbe les calculs.

In [ ]:
patients  = pd.read_excel(PATIENTS_XLS)
hospitals = pd.read_excel(HOSPIT_XLS)
feasib    = pd.read_csv(FEASIB_CSV, index_col='patient_id').astype(bool)

# Nettoyage numérique
for col in PATIENT_FEATURES:
    if col in patients.columns:
        patients[col] = pd.to_numeric(patients[col], errors='coerce').fillna(0)
for col in HOSP_RES_COLS:
    if col in hospitals.columns:
        hospitals[col] = pd.to_numeric(hospitals[col], errors='coerce').fillna(0)

print(f'Patients: {patients.shape}, Hôpitaux: {hospitals.shape}, Faisabilité: {feasib.shape}')

## 4. Génération des Labels d'Entraînement

Puisqu'il n'existe pas d'historique réel d'allocation, les labels sont générés par une **règle d'allocation optimale** :

```
Pour chaque patient :
  1. Garder uniquement les hôpitaux faisables (feasibility_matrix = True)
  2. Calculer un score = resource_score(patient, hôpital) / (1 + distance/10)
  3. L'hôpital avec le meilleur score devient le label positif (1)
```

**`haversine`** : calcule la distance réelle (en km) entre le patient et l'hôpital via les coordonnées GPS.

**`resource_score`** : mesure à quel point les ressources d'un hôpital couvrent les besoins du patient (ratio moyen, plafonné à 1.0).

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1,p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1); dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a), math.sqrt(1-a))

def resource_score(patient, hosp):
    """Score de compatibilité ressources (ratio dispo/besoin moyen)."""
    scores = []
    for spec in RESOURCE_MAPPINGS.values():
        need = float(patient.get(spec['p'], 0) or 0)
        cap  = float(hosp.get(spec['h'], 0) or 0)
        if need > 0:
            scores.append(min(cap / need, 1.0))
    return float(np.mean(scores)) if scores else 1.0

def optimal_hospital(patient, hospitals_df, feas_row):
    """Hôpital optimal : faisable + meilleur score ressources / distance."""
    best, best_score = None, -1
    for _, h in hospitals_df.iterrows():
        hname = h['Hôpital']
        if not feas_row.get(hname, False):
            continue
        dist = haversine(patient['Latitude'], patient['Longitude'], h['Lat'], h['Long'])
        sc = resource_score(patient, h) / (1 + dist / 10)
        if sc > best_score:
            best_score, best = sc, hname
    return best

print('Génération des labels...')
labels = []
for _, p in patients.iterrows():
    pid = p['ID']
    frow = feasib.loc[pid].to_dict() if pid in feasib.index else {}
    labels.append(optimal_hospital(p, hospitals, frow))
patients['Hopital_Optimal'] = labels
print(patients['Hopital_Optimal'].value_counts())

## 5. Construction du Dataset : Approche Pair-wise

Le DNN apprend à **scorer chaque paire (patient, hôpital)** individuellement.

Pour chaque patient (1 000), on crée **22 paires** — une par hôpital :
- **Label = 1** : hôpital optimal (un seul par patient)
- **Label = 0** : les 21 autres hôpitaux

Cela produit **22 000 exemples** d'entraînement avec un déséquilibre naturel de 1/22 (~4.5%).

> Ce déséquilibre sera compensé par une pondération de la fonction de perte à l'étape suivante.

In [ ]:
print('Construction des paires (patient, hôpital)...')
pairs_p, pairs_h, pairs_y = [], [], []

for _, p in patients.iterrows():
    p_vec = [float(p.get(f, 0) or 0) for f in PATIENT_FEATURES]
    optimal = p['Hopital_Optimal']
    for _, h in hospitals.iterrows():
        hname = h['Hôpital']
        dist  = haversine(p['Latitude'], p['Longitude'], h['Lat'], h['Long'])
        h_vec = [float(h.get(c, 0) or 0) for c in HOSP_RES_COLS] + [dist]
        pairs_p.append(p_vec)
        pairs_h.append(h_vec)
        pairs_y.append(1 if hname == optimal else 0)

pairs_p = np.array(pairs_p, dtype=np.float32)
pairs_h = np.array(pairs_h, dtype=np.float32)
pairs_y = np.array(pairs_y, dtype=np.float32)
print(f'Total paires: {len(pairs_y):,}  |  positives: {pairs_y.sum():.0f}  |  ratio: {pairs_y.mean()*100:.1f}%')

## 6. Normalisation & Dataset PyTorch

**Deux scalers séparés** sont utilisés (et sauvegardés pour l'inférence) :
- `scaler_p` : normalise les features patients (moyenne=0, écart-type=1)
- `scaler_h` : normalise les features hôpitaux + distance

**Split 80/20** stratifié sur les labels pour conserver le ratio positif/négatif dans train et test.

`PairDataset` est la classe PyTorch `Dataset` qui encapsule les tenseurs pour le `DataLoader` (mini-batches de 512).

In [ ]:
scaler_p = StandardScaler().fit(pairs_p)
scaler_h = StandardScaler().fit(pairs_h)
X_p = scaler_p.transform(pairs_p)
X_h = scaler_h.transform(pairs_h)
X   = np.hstack([X_p, X_h]).astype(np.float32)

# Sauvegarde des scalers
with open(OUT_DIR/'scaler_p.pkl','wb') as f: pickle.dump(scaler_p, f)
with open(OUT_DIR/'scaler_h.pkl','wb') as f: pickle.dump(scaler_h, f)

# Encodage label
le = LabelEncoder().fit(hospitals['Hôpital'])
with open(OUT_DIR/'label_encoder.pkl','wb') as f: pickle.dump(le, f)

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

idx = np.arange(len(pairs_y))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=pairs_y)
train_ds = PairDataset(X[tr_idx], pairs_y[tr_idx])
test_ds  = PairDataset(X[te_idx], pairs_y[te_idx])
train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=512)
print(f'Train: {len(train_ds):,}  |  Test: {len(test_ds):,}')

## 7. Architecture du Réseau de Neurones (DNN)

```
Input (54)  →  Dense(256) + BatchNorm + ReLU + Dropout(30%)
            →  Dense(128) + BatchNorm + ReLU + Dropout(30%)
            →  Dense(64)  + BatchNorm + ReLU + Dropout(20%)
            →  Dense(32)  + ReLU
            →  Dense(1)   + Sigmoid  →  score ∈ [0,1]
```

**Choix de conception :**
- `BatchNorm1d` : stabilise l'entraînement en normalisant les activations internes
- `Dropout` : régularisation pour éviter le surapprentissage (dataset de taille modeste)
- `Sigmoid` en sortie : interprétable comme une **probabilité de compatibilité**

In [ ]:
class PatientAllocationDNN(nn.Module):
    """DNN pair-wise : score de compatibilité (patient, hôpital) → [0,1]."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64),       nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear( 64,  32),       nn.ReLU(),
            nn.Linear( 32,   1),       nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

model = PatientAllocationDNN(INPUT_DIM).to(DEVICE)
print(model)
total = sum(p.numel() for p in model.parameters())
print(f'Paramètres: {total:,}')

## 8. Entraînement du DNN

**Stratégie pour le déséquilibre de classes (1 positif / 21 négatifs) :**
- `BCEWithLogitsLoss(pos_weight=21.0)` : pénalise 21× plus les faux négatifs
- Le Sigmoid est retiré de l'architecture pour l'entraînement (numériquement plus stable)

**Optimiseur & Scheduler :**
- `AdamW` (lr=0.001, weight_decay=1e-4) : Adam avec régularisation L2
- `ReduceLROnPlateau` (patience=3) : divise le taux d'apprentissage par 2 si la val_loss stagne

**Sauvegarde du meilleur état** : seul le modèle avec la val_loss minimale est conservé (`dnn_weights.pt`).

In [ ]:
# Pondération pour déséquilibre de classes (1 positif pour 21 négatifs)
pos_weight = torch.tensor([21.0]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Utiliser les logits (retirer le Sigmoid final pour BCEWithLogitsLoss)
class PatientAllocationDNN_Train(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64),       nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear( 64,  32),       nn.ReLU(),
            nn.Linear( 32,   1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)
    def predict_proba(self, x): return torch.sigmoid(self.forward(x))

model = PatientAllocationDNN_Train(INPUT_DIM).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 40
history = {'train_loss':[], 'val_loss':[], 'val_acc':[]}

best_val, best_state = float('inf'), None

for epoch in range(1, EPOCHS+1):
    # --- Train ---
    model.train()
    tr_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    # --- Val ---
    model.eval()
    val_loss, preds, truths = 0.0, [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            val_loss += criterion(logits, yb).item() * len(yb)
            preds.extend((torch.sigmoid(logits) > 0.5).cpu().numpy())
            truths.extend(yb.cpu().numpy())
    val_loss /= len(test_ds)
    val_acc   = accuracy_score(truths, preds)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val   = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0:
        print(f'Ep {epoch:02d}/{EPOCHS} | tr_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc*100:.1f}%')

model.load_state_dict(best_state)
torch.save(model.state_dict(), OUT_DIR/'dnn_weights.pt')
print('Meilleur modèle sauvegardé.')

## 9. Évaluation du Modèle

Deux niveaux d'évaluation :

**1. Courbes d'entraînement**
- `train_loss` vs `val_loss` : vérifie l'absence de surapprentissage
- `val_acc` : précision globale sur les paires test

**2. Métriques par patient (plus significatives)**
- **Top-1 Accuracy** : l'hôpital optimal est-il le 1er suggéré par le DNN ?
- **Top-3 Accuracy** : l'hôpital optimal est-il dans les 3 premières suggestions ?

> L'objectif est Top-1 ≥ 75% et Top-3 ≥ 95%.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['train_loss'], label='Train loss')
axes[0].plot(history['val_loss'],   label='Val loss')
axes[0].set_title('Courbe de perte'); axes[0].legend()
axes[1].plot(history['val_acc'])
axes[1].set_title('Précision validation'); axes[1].set_ylabel('Accuracy')
plt.tight_layout(); plt.savefig(OUT_DIR/'training_curves.png', dpi=120)
plt.show()

# Rapport par patient : top-1 et top-3 accuracy
model.eval()
n_pat = len(patients)
n_h   = len(hospitals)
top1_ok = top3_ok = 0

for i, (_, p) in enumerate(patients.iterrows()):
    p_vec = np.array([float(p.get(f,0) or 0) for f in PATIENT_FEATURES], dtype=np.float32)
    scores = []
    for _, h in hospitals.iterrows():
        dist  = haversine(p['Latitude'], p['Longitude'], h['Lat'], h['Long'])
        h_vec = np.array([float(h.get(c,0) or 0) for c in HOSP_RES_COLS] + [dist], dtype=np.float32)
        p_n = scaler_p.transform([p_vec])[0]
        h_n = scaler_h.transform([h_vec])[0]
        x   = torch.FloatTensor(np.concatenate([p_n, h_n])).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): scores.append(torch.sigmoid(model(x)).item())
    ranked = np.argsort(scores)[::-1]
    true_h = p['Hopital_Optimal']
    true_i = list(hospitals['Hôpital']).index(true_h) if true_h else -1
    if true_i >= 0:
        if ranked[0] == true_i:  top1_ok += 1
        if true_i in ranked[:3]: top3_ok += 1

print(f'Top-1 Accuracy : {top1_ok/n_pat*100:.1f}%')
print(f'Top-3 Accuracy : {top3_ok/n_pat*100:.1f}%')

## 10. Fonction d'Inférence

**`predict_hospital(patient_dict, hospitals_df, hospital_states=None)`**

Retourne la liste des 22 hôpitaux triés par score décroissant.

- En mode **statique** (défaut) : utilise les capacités fixes de `Book1.xlsx`
- En mode **dynamique** : utilise `hospital_states` (ressources actuelles transmises par `AllocationEngine`)

> Cette fonction est appelée à chaque arrivée de patient dans le système temps-réel.

In [ ]:
def predict_hospital(patient_dict, hospitals_df, hospital_states=None):
    """
    Prédit l'hôpital optimal pour un patient.
    patient_dict  : dict avec les colonnes patients
    hospital_states : dict {nom: vecteur_ressources_actuel} (optionnel, sinon Book1)
    """
    model.eval()
    p_vec = np.array([float(patient_dict.get(f,0) or 0) for f in PATIENT_FEATURES], dtype=np.float32)
    results = []
    for _, h in hospitals_df.iterrows():
        hname = h['Hôpital']
        if hospital_states and hname in hospital_states:
            res_vec = hospital_states[hname]
        else:
            res_vec = [float(h.get(c,0) or 0) for c in HOSP_RES_COLS]
        dist  = haversine(patient_dict.get('Latitude',0), patient_dict.get('Longitude',0), h['Lat'], h['Long'])
        h_vec = np.array(res_vec + [dist], dtype=np.float32)
        p_n   = scaler_p.transform([p_vec])[0]
        h_n   = scaler_h.transform([h_vec])[0]
        x     = torch.FloatTensor(np.concatenate([p_n, h_n])).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): score = torch.sigmoid(model(x)).item()
        results.append({'hospital': hname, 'score': score})
    results.sort(key=lambda r: r['score'], reverse=True)
    return results

# Test rapide
test_patient = patients.iloc[0].to_dict()
recs = predict_hospital(test_patient, hospitals)
print('Top-5 hôpitaux pour le patient 1 :')
for r in recs[:5]:
    print(f"  {r['score']:.4f}  {r['hospital']}")